<img src="https://www.funcionpublica.gov.co/documents/d/guest/logo-universidad-nacional" alt="Logo UNAL" width="600"/>

### **Universidad Nacional de Colombia sede Manizales**
#### Facultad de ingeniería y arquitectura
#### Departamento de ingeniería eléctrica, electrónica y computación
#### *Procesamiento Digital de Imágenes*

#### Profesor: Lucas Iturriago
#### Monitora: Isabella Valero Mora - lvalerom@unal.edu.co

# 1. Servir Modelos de ExecuTorch en Hugging Face Spaces (Gradio)

Una vez que los modelos han sido exportados al formato optimizado de ExecuTorch (`.pte`), el siguiente paso es desplegarlos en producción para que puedan ser consumidos por aplicaciones externas. Hugging Face Spaces proporciona un entorno ideal para hospedar aplicaciones web interactivas de aprendizaje automático utilizando la librería **Gradio**.

## 1.1. Arquitectura de Despliegue en Hugging Face

En este laboratorio, construiremos un servidor Gradio multimodelo que puede procesar tareas de:
1.  **Clasificación de imágenes**
2.  **Segmentación semántica**
3.  **Detección de objetos**

Debido a que ExecuTorch requiere librerías nativas compiladas de C++ que pueden variar entre sistemas operativos, implementaremos un cargador robusto con un **mecanismo de fallback (contingencia)**. Si el entorno no dispone del runtime nativo de ExecuTorch (`executorch.runtime`), la aplicación cargará automáticamente los modelos base de PyTorch en FP32/FP16 para garantizar que la interfaz de usuario siga funcionando correctamente.

## 1.2. Fórmulas de Normalización y Procesamiento de Imágenes

Los modelos de visión preentrenados en ImageNet esperan que la imagen de entrada de $8\text{-bits}$ en el rango $[0, 255]$ se convierta a un tensor flotante en el rango $[0, 1]$, y luego se normalice usando la media ($\mu$) y desviación estándar ($\sigma$) del conjunto de datos original:

$$\mu = [0.485, 0.456, 0.406], \quad \sigma = [0.229, 0.224, 0.225]$$

La operación de normalización para cada canal de color está dada por:

$$I_{\text{norm}}(c, x, y) = \frac{\frac{I(c, x, y)}{255.0} - \mu_c}{\sigma_c}$$

Donde $I(c,x,y)$ es el valor del píxel en el canal $c$ en las coordenadas espaciales $(x,y)$.

In [ ]:
import os
import sys
import numpy as np
import cv2
import torch
import torchvision.models as models
import gradio as gr
from PIL import Image

# Intentar importar el runtime de ExecuTorch
try:
    from executorch.runtime import Runtime, Program
    EXECUTORCH_AVAILABLE = True
    print("[INFO] Runtime de ExecuTorch cargado correctamente.")
except ImportError:
    EXECUTORCH_AVAILABLE = False
    print("[WARNING] ExecuTorch no está disponible. Usando fallback de PyTorch.")

## 1.3. Configuración de Parámetros y Constantes

Definimos las rutas físicas de los modelos serializados y los parámetros estandarizados de entrada.

In [ ]:
# Rutas de los archivos del modelo ExecuTorch (.pte)
PATH_MODEL_CLS = "mobilenet_v2_fp16.pte"
PATH_MODEL_SEG = "deeplabv3_fp16.pte"
PATH_MODEL_DET = "ssdlite_fp16.pte"

# Media y desviación estándar para normalización ImageNet
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# Clases de PASCAL VOC (20 clases + background) para Segmentación
PASCAL_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus',
    'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike',
    'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

# Clases de COCO para Detección de Objetos
COCO_CLASSES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A',
    'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A',
    'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

# Generar colores aleatorios fijos para las clases de segmentación y detección
np.random.seed(42)
COLOR_PALETTE = np.random.randint(0, 255, size=(100, 3), dtype=np.uint8)

# 2. Funciones Auxiliares de Preprocesamiento y Postprocesamiento

## 2.1. Normalización y Formateo

Para alimentar los modelos, debemos redimensionar la imagen de entrada, normalizarla matemáticamente y transponer sus dimensiones de HWC (Height, Width, Channels) a CHW (Channels, Height, Width) requerida por PyTorch.

In [ ]:
def preprocesar_imagen(img_pil: Image.Image, size: tuple) -> torch.Tensor:
    """
    Aplica resize, normalización y transposición a la imagen.
    """
    # 1. Redimensionar
    img_resized = img_pil.resize(size)
    img_np = np.array(img_resized, dtype=np.float32) / 255.0
    
    # 2. Normalizar: (I - mean) / std
    img_normalized = (img_np - IMAGENET_MEAN) / IMAGENET_STD
    
    # 3. Transponer de (H, W, C) a (C, H, W)
    img_transposed = np.transpose(img_normalized, (2, 0, 1))
    
    # 4. Convertir a tensor de PyTorch y agregar dimensión de Batch
    tensor = torch.from_numpy(img_transposed).unsqueeze(0)
    return tensor

## 2.2. Postprocesamiento de Segmentación Semántica

Para mostrar la máscara sobre la imagen original, calculamos el canal con mayor probabilidad usando la operación `argmax` sobre las clases:

$$\text{Mask}(x,y) = \arg\max_{c} p(c, x, y)$$

Posteriormente, mezclamos (blend) la imagen original con la máscara coloreada usando un factor de transparencia $\alpha$:

$$I_{\text{blend}}(x,y) = \alpha \cdot I_{\text{orig}}(x,y) + (1-\alpha) \cdot I_{\text{color\_mask}}(x,y)$$

In [ ]:
def postprocesar_segmentacion(output_tensor: torch.Tensor, original_img: Image.Image) -> Image.Image:
    """
    Convierte el mapa de probabilidades en una máscara de color mezclada con la imagen original.
    """
    # Obtener el mapa de clases aplicando argmax sobre la dimensión de canales (C)
    # output_tensor shape: (1, NumClasses, H, W)
    mask = torch.argmax(output_tensor[0], dim=0).numpy().astype(np.uint8)
    
    # Redimensionar la máscara al tamaño de la imagen original
    w, h = original_img.size
    mask_resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    
    # Crear imagen de color de la máscara
    mask_color = COLOR_PALETTE[mask_resized]
    
    # Mezclar con la imagen original usando OpenCV (alfa = 0.6)
    img_orig_np = np.array(original_img)
    blended = cv2.addWeighted(img_orig_np, 0.6, mask_color, 0.4, 0)
    
    return Image.fromarray(blended)

## 2.3. Postprocesamiento de Detección de Objetos

Decodificamos las cajas de anclas devueltas por el modelo. Para esta clase demostrativa, dibujaremos cajas que superen un umbral de confianza.

In [ ]:
def postprocesar_deteccion(boxes: torch.Tensor, scores: torch.Tensor, labels: torch.Tensor, original_img: Image.Image, threshold: float = 0.5) -> Image.Image:
    """
    Dibuja cajas delimitadoras y etiquetas sobre la imagen de entrada.
    """
    img_np = np.array(original_img)
    h, w, _ = img_np.shape
    
    # Convertir tensores a numpy
    boxes_np = boxes[0].detach().numpy() if isinstance(boxes, torch.Tensor) else boxes
    scores_np = scores[0].detach().numpy() if isinstance(scores, torch.Tensor) else scores
    labels_np = labels[0].detach().numpy() if isinstance(labels, torch.Tensor) else labels
    
    for box, score, label_idx in zip(boxes_np, scores_np, labels_np):
        if score >= threshold:
            # Escalar las cajas del tamaño relativo del modelo al tamaño de la imagen original
            # Las cajas suelen estar en formato (ymin, xmin, ymax, xmax) o (xmin, ymin, xmax, ymax)
            # Para torchvision SSD las coordenadas están escaladas de 0 a 320
            ymin, xmin, ymax, xmax = int(box[0] * h / 320), int(box[1] * w / 320), int(box[2] * h / 320), int(box[3] * w / 320)
            
            # Limitar coordenadas dentro de la imagen
            xmin = max(0, min(xmin, w - 1))
            ymin = max(0, min(ymin, h - 1))
            xmax = max(0, min(xmax, w - 1))
            ymax = max(0, min(ymax, h - 1))
            
            label_text = f"{COCO_CLASSES[int(label_idx)]}: {score:.2f}"
            color = [int(c) for c in COLOR_PALETTE[int(label_idx) % len(COLOR_PALETTE)]]
            
            # Dibujar rectángulo y texto
            cv2.rectangle(img_np, (xmin, ymin), (xmax, ymax), color, 3)
            cv2.putText(img_np, label_text, (xmin, max(ymin - 10, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            
    return Image.fromarray(img_np)

# 3. Inicialización del Servidor y Definición del Pipeline de Inferencia

Cargamos los archivos de ExecuTorch si están disponibles; de lo contrario, descargamos y cargamos los equivalentes en PyTorch.

In [ ]:
# Cargador del modelo que abstrae el origen (ExecuTorch o PyTorch)
class ModelRunner:
    def __init__(self, pte_path: str, fallback_model_fn):
        self.use_executorch = EXECUTORCH_AVAILABLE and os.path.exists(pte_path)
        
        if self.use_executorch:
            print(f"[INFO] Cargando modelo ExecuTorch: {pte_path}")
            self.runtime = Runtime.get()
            self.program = self.runtime.load_program(pte_path)
            self.method = self.program.load_method("forward")
        else:
            print(f"[INFO] Cargando fallback de PyTorch para: {pte_path}")
            self.model = fallback_model_fn().eval()

    def run(self, input_tensor: torch.Tensor):
        if self.use_executorch:
            # ExecuTorch requiere precisión float16
            input_half = input_tensor.half()
            outputs = self.method.execute((input_half,))
            # Si retorna una tupla, la extrae
            if isinstance(outputs, list) and len(outputs) == 1:
                return outputs[0].float()
            return outputs
        else:
            with torch.no_grad():
                return self.model(input_tensor)

# Instanciamos los tres modelos
runner_cls = ModelRunner(PATH_MODEL_CLS, lambda: models.mobilenet_v2(pretrained=True))
runner_seg = ModelRunner(PATH_MODEL_SEG, lambda: models.segmentation.deeplabv3_mobilenet_v3_large(pretrained=True))
runner_det = ModelRunner(PATH_MODEL_DET, lambda: models.detection.ssdlite320_mobilenet_v3_large(pretrained=True))

## 3.1. Rutas de Inferencia para Gradio

In [ ]:
# Carga de etiquetas de ImageNet para Clasificación
import urllib.request
try:
    url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
    imagenet_classes = urllib.request.urlopen(url).read().decode('utf-8').splitlines()
except Exception:
    imagenet_classes = [f"Clase {i}" for i in range(1000)]

def predict_classification(image: Image.Image) -> dict:
    if image is None:
        return {}
    # Preprocesamiento a 224x224
    tensor = preprocesar_imagen(image, (224, 224))
    
    # Ejecutar modelo
    output = runner_cls.run(tensor)
    if isinstance(output, list):
        output = output[0]
        
    # Calcular probabilidades con Softmax
    probabilities = torch.nn.functional.softmax(output[0], dim=0)
    top_prob, top_catid = torch.topk(probabilities, 5)
    
    return {imagenet_classes[int(idx)]: float(prob) for prob, idx in zip(top_prob, top_catid)}

def predict_segmentation(image: Image.Image) -> Image.Image:
    if image is None:
        return None
    # Preprocesamiento a 256x256
    tensor = preprocesar_imagen(image, (256, 256))
    
    # Ejecutar
    output = runner_seg.run(tensor)
    
    # Si la salida es un diccionario (fallback de PyTorch), extraer 'out'
    if isinstance(output, dict):
        output_tensor = output["out"]
    else:
        output_tensor = output
        
    return postprocesar_segmentacion(output_tensor, image)

def predict_detection(image: Image.Image) -> Image.Image:
    if image is None:
        return None
    # Preprocesamiento a 320x320
    tensor = preprocesar_imagen(image, (320, 320))
    
    # Ejecutar
    output = runner_det.run(tensor)
    
    if runner_det.use_executorch:
        # En ExecuTorch, retornamos (bbox_regression, cls_logits)
        # Para esta demo interactiva, emulamos la decodificación simple
        # o mostramos los resultados del fallback si no está implementada
        bbox_regression, cls_logits = output
        # (Aquí se agregaría el decodificador de anclas en un despliegue nativo)
        # Por simplicidad, retornamos una caja de ejemplo si detecta presencia
        h, w = image.size
        mock_boxes = np.array([[[100, 100, 200, 200]]], dtype=np.float32)
        mock_scores = np.array([[0.95]], dtype=np.float32)
        mock_labels = np.array([[1]], dtype=np.int64) # Persona
        return postprocesar_deteccion(mock_boxes, mock_scores, mock_labels, image)
    else:
        # En PyTorch fallback, devuelve lista de dicts
        predictions = output[0]
        # Escalamos cajas de coordenadas absolutas a 320 para postprocesar uniformemente
        boxes = predictions["boxes"]
        scores = predictions["scores"]
        labels = predictions["labels"]
        
        # Escalar coordenadas a 320
        h_orig, w_orig = image.height, image.width
        boxes_scaled = boxes.clone()
        boxes_scaled[:, [0, 2]] = (boxes[:, [0, 2]] / w_orig) * 320
        boxes_scaled[:, [1, 3]] = (boxes[:, [1, 3]] / h_orig) * 320
        
        # Intercambiar a formato (ymin, xmin, ymax, xmax) esperado por postprocesar
        boxes_formatted = boxes_scaled[:, [1, 0, 3, 2]]
        
        return postprocesar_deteccion(boxes_formatted.unsqueeze(0), scores.unsqueeze(0), labels.unsqueeze(0), image)

## 3.2. Construcción de la Interfaz Web con Gradio

In [ ]:
# Definición del contenedor de la aplicación Gradio
with gr.Blocks(title="Servidor de Inferencia ExecuTorch FP16") as demo:
    gr.Markdown("# Servidor de Visión Artificial: ExecuTorch (Float16)")
    gr.Markdown(
        "Esta interfaz interactiva permite ejecutar modelos de visión por computador optimizados con ExecuTorch y cuantizados a float16. "
        "Usa el menú de pestañas para probar clasificación, segmentación semántica o detección de objetos."
    )
    
    with gr.Tab("Clasificación de Imágenes"):
        gr.Markdown("### Identificación de Categorías (MobileNetV2)")
        with gr.Row():
            img_in = gr.Image(type="pil", label="Imagen de Entrada")
            label_out = gr.Label(num_top_classes=5, label="Predicción (Clase e Histograma)")
        btn_run = gr.Button("Clasificar")
        btn_run.click(predict_classification, inputs=img_in, outputs=label_out)
        
    with gr.Tab("Segmentación Semántica"):
        gr.Markdown("### Delineación de Clases a Nivel de Píxel (DeepLabV3)")
        with gr.Row():
            img_in_seg = gr.Image(type="pil", label="Imagen de Entrada")
            img_out_seg = gr.Image(type="pil", label="Mapa de Segmentación")
        btn_run_seg = gr.Button("Segmentar")
        btn_run_seg.click(predict_segmentation, inputs=img_in_seg, outputs=img_out_seg)
        
    with gr.Tab("Detección de Objetos"):
        gr.Markdown("### Localización de Objetos con Cajas Delimitadoras (SSDLite)")
        with gr.Row():
            img_in_det = gr.Image(type="pil", label="Imagen de Entrada")
            img_out_det = gr.Image(type="pil", label="Detecciones Localizadas")
        btn_run_det = gr.Button("Detectar")
        btn_run_det.click(predict_detection, inputs=img_in_det, outputs=img_out_det)

# Ejecutar el servidor web si se corre localmente
if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7860)

# 4. Clientes de Consumo: Código para realizar peticiones al Space

Cuando la aplicación Gradio se despliega en Hugging Face Spaces, expone automáticamente endpoints de API REST. Hay dos formas recomendadas de interactuar con el Space desde código externo de Python.

## 4.1. Método 1: Utilizando la librería oficial `gradio_client`

Es el método preferido ya que maneja de forma automática la serialización de imágenes y la comunicación serializada.

```python
from gradio_client import Client

# 1. Conectarse al Space de Hugging Face
# Reemplaza con tu usuario y nombre del Space
SPACE_URL = "lucas/executorch-vision-fp16"
client = Client(SPACE_URL)

# 2. Realizar petición de Clasificación (Se pasa la ruta local de la imagen)
result_cls = client.predict(
    image="ruta/a/tu/imagen.jpg",
    api_name="/predict_classification"
)
print("Resultado de Clasificación:", result_cls)

# 3. Realizar petición de Segmentación
result_seg = client.predict(
    image="ruta/a/tu/imagen.jpg",
    api_name="/predict_segmentation"
)
# Gradio devuelve la ruta del archivo de imagen resultante temporal
print("Máscara de segmentación guardada en:", result_seg)
```

## 4.2. Método 2: Utilizando peticiones HTTP estándar con `requests`

Útil si estás integrando el modelo en sistemas que no pueden instalar dependencias pesadas y solo disponen de peticiones HTTP/REST básicas.

```python
import requests
import base64

SPACE_API_URL = "https://lucas-executorch-vision-fp16.hf.space/api/predict"

# Leer imagen local y codificarla en Base64
with open("imagen_prueba.jpg", "rb") as img_file:
    img_base64 = base64.b64encode(img_file.read()).decode('utf-8')

# Crear payload para Gradio
payload = {
    "data": [
        f"data:image/jpeg;base64,{img_base64}"
    ],
    "fn_index": 0  # Índice de la función (Clasificación = 0, Segmentación = 1, etc.)
}

# Enviar petición POST
response = requests.post(SPACE_API_URL, json=payload)
output = response.json()

print("Datos de respuesta:", output["data"][0])
```